# Model Metrics Verification

Validate the project metric helpers on real THINGS-EEG2 validation and test dataloaders:

- validation split: compare `open_clip_train.train.get_clip_metrics` with an independent full-retrieval reference
- test split: compare `src.metrics.retrieval.get_kway_metrics` with an independent k-way retrieval reference

This notebook verifies metric standards and calling conventions. It does not load a trained EEG checkpoint.

In [ ]:
import os
import sys
from pathlib import Path

import torch
import torch.nn.functional as F
from omegaconf import OmegaConf
from rich.console import Console
from rich.panel import Panel
from rich.pretty import Pretty
from rich.table import Table

console = Console()

In [ ]:
def find_project_root(start=Path.cwd()):
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "configs").exists() and (candidate / "src").exists():
            return candidate
    return start


def load_paths_config(root):
    os.environ.setdefault("PROJECT_ROOT", str(root))
    paths = OmegaConf.load(root / "configs" / "paths" / "default.yaml")
    return OmegaConf.create({"paths": paths})


def config_path(paths_config, key):
    value = OmegaConf.select(paths_config, f"paths.{key}")
    return Path(str(value)).expanduser()


def to_plain_list(value):
    if value is None:
        return None
    return list(value)


ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from open_clip_train.train import get_clip_metrics as open_clip_get_clip_metrics

from src.data.thingseeg2_datamodule import ThingsEEG2DataModule
from src.metrics.retrieval import get_kway_metrics
from src.utils.config_resolvers import resolve_clip_model_id

paths_config = load_paths_config(ROOT)
data_config = OmegaConf.load(ROOT / "configs" / "data" / "thingseeg2.yaml")

model_name, model_id = resolve_clip_model_id(data_config.model_name, data_config.model_id)

datamodule = ThingsEEG2DataModule(
    eeg_data_dir=config_path(paths_config, "thingseeg2_preprocessed_dir"),
    clip_features_dir=config_path(paths_config, "thingseeg2_clip_features_dir"),
    subjects=to_plain_list(data_config.subjects),
    experiment_setting=data_config.experiment_setting,
    train_val_split=to_plain_list(data_config.train_val_split),
    train_batch_size=int(data_config.train_batch_size),
    val_batch_size=int(data_config.val_batch_size),
    test_batch_size=int(data_config.test_batch_size),
    num_workers=0,
    pin_memory=False,
    drop_last=bool(data_config.drop_last),
    k_fold=data_config.k_fold,
    fold_idx=int(data_config.fold_idx),
    average_reps=bool(data_config.average_reps),
    selected_channels=to_plain_list(data_config.selected_channels),
    model_name=model_name,
    model_id=model_id,
    feature_mode=data_config.feature_mode,
)
datamodule.setup()

console.print(Panel.fit(str(ROOT), title="Project Root", border_style="cyan"))
console.print(
    Panel(
        Pretty(datamodule.describe(include_batch=True)), title="DataModule", border_style="green"
    )
)

## Shared Helpers

The dataloader produces one CLIP image feature per sample. For validation, duplicate image rows can appear when a split contains repeated subjects or repetitions, so the full-retrieval check keeps the first sample for each `image_index` before computing rank metrics. The dataset `label` field is not used as the unique feature-row key because it can represent a repeated concept/category label.

In [ ]:
def collect_features(dataloader, feature_key="image_features", max_batches=None):
    features = []
    labels = []
    image_indices = []

    for batch_idx, batch in enumerate(dataloader):
        if max_batches is not None and batch_idx >= max_batches:
            break
        features.append(batch[feature_key].detach().cpu().float())
        labels.append(batch["label"].detach().cpu().long())
        image_indices.append(batch["image_index"].detach().cpu().long())

    return {
        "features": torch.cat(features, dim=0),
        "labels": torch.cat(labels, dim=0),
        "image_indices": torch.cat(image_indices, dim=0),
    }


def first_unique_by_key(features, keys):
    seen = set()
    keep_indices = []
    for idx, key in enumerate(keys.tolist()):
        if key in seen:
            continue
        seen.add(key)
        keep_indices.append(idx)
    keep_indices = torch.tensor(keep_indices, dtype=torch.long)
    return features[keep_indices], keys[keep_indices]


def recall_at_k_from_logits(logits, k):
    n = logits.shape[0]
    k = min(k, n)
    target = torch.arange(n, device=logits.device)
    ranking = logits.argsort(dim=-1, descending=True)
    matches = ranking[:, :k].eq(target[:, None])
    return matches.any(dim=-1).float().mean()


def reference_clip_metrics(query_features, target_features, logit_scale=1.0):
    query_features = F.normalize(query_features.float(), dim=-1)
    target_features = F.normalize(target_features.float(), dim=-1)
    logits = logit_scale * query_features @ target_features.T
    reverse_logits = logits.T

    return {
        "image_to_text_R@1": recall_at_k_from_logits(logits, 1),
        "image_to_text_R@5": recall_at_k_from_logits(logits, 5),
        "image_to_text_R@10": recall_at_k_from_logits(logits, 10),
        "text_to_image_R@1": recall_at_k_from_logits(reverse_logits, 1),
        "text_to_image_R@5": recall_at_k_from_logits(reverse_logits, 5),
        "text_to_image_R@10": recall_at_k_from_logits(reverse_logits, 10),
    }


def reference_kway_metrics(
    query_features, query_labels, candidate_features, logit_scale=1.0, k=200
):
    query_features = F.normalize(query_features.float(), dim=-1)
    candidate_features = F.normalize(candidate_features.float(), dim=-1)
    query_labels = query_labels.long()

    groups, queries, _ = query_features.shape
    num_candidates = candidate_features.shape[0]
    retrieval_k = min(k, num_candidates)
    top1_hits = []
    top5_hits = []

    for group_idx in range(groups):
        for query_idx in range(queries):
            label = int(query_labels[group_idx, query_idx].item())
            candidates = [label]
            offset = 1
            while len(candidates) < retrieval_k:
                candidates.append((label + offset) % num_candidates)
                offset += 1

            candidate_index = torch.tensor(candidates, dtype=torch.long)
            logits = logit_scale * (
                candidate_features[candidate_index] @ query_features[group_idx, query_idx]
            )
            ranking = logits.argsort(descending=True)
            true_position = int((candidate_index == label).nonzero(as_tuple=False)[0].item())
            top1_hits.append(bool((ranking[:1] == true_position).any().item()))
            if retrieval_k >= 5:
                top5_hits.append(bool((ranking[:5] == true_position).any().item()))

    metrics = {"top1_acc": torch.tensor(top1_hits, dtype=torch.float32).mean()}
    if retrieval_k >= 5:
        metrics["top5_acc"] = torch.tensor(top5_hits, dtype=torch.float32).mean()
    return metrics


def as_tensor(value):
    return torch.as_tensor(value, dtype=torch.float32)


def render_comparison(title, rows, atol=1e-6):
    table = Table(title=title, show_lines=True)
    table.add_column("Metric", style="bold cyan")
    table.add_column("Project", justify="right")
    table.add_column("Reference", justify="right")
    table.add_column("Abs Diff", justify="right")
    table.add_column("Match", justify="center")

    for metric, project_value, reference_value in rows:
        project_tensor = as_tensor(project_value)
        reference_tensor = as_tensor(reference_value)
        diff = (project_tensor - reference_tensor).abs()
        match = torch.allclose(project_tensor, reference_tensor, atol=atol, rtol=0.0)
        table.add_row(
            metric,
            f"{project_tensor.item():.8f}",
            f"{reference_tensor.item():.8f}",
            f"{diff.item():.3g}",
            "yes" if match else "NO",
        )
        assert (
            match
        ), f"{metric} mismatch: project={project_tensor.item()} reference={reference_tensor.item()}"

    console.print(table)

## Validation Split: OpenCLIP Full Retrieval Metrics

This checks that `open_clip_train.train.get_clip_metrics` agrees with an independent full similarity-matrix reference on real validation features.

In [ ]:
VAL_MAX_BATCHES = None
LOGIT_SCALE = torch.tensor(1.0)

val_data = collect_features(datamodule.val_dataloader(), max_batches=VAL_MAX_BATCHES)
val_features, val_image_indices = first_unique_by_key(
    val_data["features"], val_data["image_indices"]
)

console.print(
    Panel.fit(
        f"raw rows={val_data['features'].shape[0]:,} | unique image_index={val_features.shape[0]:,} | dim={val_features.shape[1]:,}",
        title="Validation Feature Set",
        border_style="cyan",
    )
)

project_val_metrics = open_clip_get_clip_metrics(val_features, val_features, LOGIT_SCALE)
reference_val_metrics = reference_clip_metrics(val_features, val_features, LOGIT_SCALE)

val_rows = [
    (metric, project_val_metrics[metric], reference_val_metrics[metric])
    for metric in (
        "image_to_text_R@1",
        "image_to_text_R@5",
        "image_to_text_R@10",
        "text_to_image_R@1",
        "text_to_image_R@5",
        "text_to_image_R@10",
    )
]
render_comparison("Validation OpenCLIP Metric Check", val_rows)

## Test Split: k-way Retrieval Metrics

This checks that `get_kway_metrics` agrees with an independent reference using real test features. It mirrors `ClipV1LitModule.on_test_epoch_end`: after collecting features, the query label is the row index in the candidate feature matrix. The candidate set follows the project standard: the true row plus the next `k - 1` candidate rows modulo the number of candidates.

In [ ]:
TEST_MAX_BATCHES = None
RETRIEVAL_K_LIST = [2, 4, 10, 200]

test_data = collect_features(datamodule.test_dataloader(), max_batches=TEST_MAX_BATCHES)
test_features = test_data["features"]

query_features = test_features.unsqueeze(0)
query_labels = torch.arange(test_features.shape[0], dtype=torch.long).unsqueeze(0)
candidate_features = test_features

console.print(
    Panel.fit(
        f"queries={query_features.shape[1]:,} | candidates={candidate_features.shape[0]:,} | dim={candidate_features.shape[1]:,}",
        title="Test Feature Set",
        border_style="cyan",
    )
)

for retrieval_k in RETRIEVAL_K_LIST:
    project_test_metrics = get_kway_metrics(
        query_features=query_features,
        query_labels=query_labels,
        candidate_features=candidate_features,
        logit_scale=LOGIT_SCALE,
        k=retrieval_k,
    )
    reference_test_metrics = reference_kway_metrics(
        query_features=query_features,
        query_labels=query_labels,
        candidate_features=candidate_features,
        logit_scale=LOGIT_SCALE,
        k=retrieval_k,
    )
    test_rows = [
        (metric, project_test_metrics[metric], reference_test_metrics[metric])
        for metric in project_test_metrics.keys()
    ]
    render_comparison(f"Test k-way Metric Check, k={retrieval_k}", test_rows)

## Controlled k-way Cases

These synthetic cases make the expected k-way behavior visible beyond the real test split, where image features compared to themselves usually produce perfect scores. Each case still compares `get_kway_metrics` against the independent reference implementation above.

In [ ]:
def run_controlled_kway_case(
    name, query_features, query_labels, candidate_features, k, expected=None
):
    project_metrics = get_kway_metrics(
        query_features=query_features,
        query_labels=query_labels,
        candidate_features=candidate_features,
        logit_scale=LOGIT_SCALE,
        k=k,
    )
    reference_metrics = reference_kway_metrics(
        query_features=query_features,
        query_labels=query_labels,
        candidate_features=candidate_features,
        logit_scale=LOGIT_SCALE,
        k=k,
    )
    rows = [
        (metric, project_metrics[metric], reference_metrics[metric])
        for metric in project_metrics.keys()
    ]
    render_comparison(f"Controlled k-way Case: {name}", rows)

    if expected is not None:
        for metric, expected_value in expected.items():
            actual = as_tensor(project_metrics[metric])
            target = torch.tensor(expected_value, dtype=torch.float32)
            assert torch.allclose(
                actual, target, atol=1e-6, rtol=0.0
            ), f"{name} {metric} expected {expected_value}, got {actual.item()}"


perfect_candidates = F.normalize(
    torch.tensor(
        [
            [1.0, 0.0, 0.0],
            [0.0, 1.0, 0.0],
            [0.0, 0.0, 1.0],
            [-1.0, 0.0, 0.0],
            [0.0, -1.0, 0.0],
            [0.0, 0.0, -1.0],
        ]
    ),
    dim=-1,
)
top1_fail_top5_success_candidates = torch.tensor(
    [
        [0.9, 0.1],
        [1.0, 0.0],
        [0.6, 0.8],
        [0.3, 0.95],
        [0.0, 1.0],
        [-1.0, 0.0],
    ]
)
top5_fail_candidates = torch.tensor(
    [
        [-1.0, 0.0],
        [1.0, 0.0],
        [0.9, 0.1],
        [0.8, 0.2],
        [0.7, 0.3],
        [0.6, 0.4],
    ]
)
mixed_candidates = torch.tensor(
    [
        [1.0, 0.0],
        [0.9, 0.1],
        [-1.0, 0.0],
        [0.0, 1.0],
        [0.0, -1.0],
        [-0.9, -0.1],
    ]
)

controlled_cases = [
    {
        "name": "perfect top1, k=6",
        "query_features": perfect_candidates[[0, 1, 2]].unsqueeze(0),
        "query_labels": torch.tensor([[0, 1, 2]]),
        "candidate_features": perfect_candidates,
        "k": 6,
        "expected": {"top1_acc": 1.0, "top5_acc": 1.0},
    },
    {
        "name": "top1 fails but top5 succeeds, k=6",
        "query_features": torch.tensor([[[1.0, 0.0]]]),
        "query_labels": torch.tensor([[0]]),
        "candidate_features": top1_fail_top5_success_candidates,
        "k": 6,
        "expected": {"top1_acc": 0.0, "top5_acc": 1.0},
    },
    {
        "name": "top5 fails when true target ranks sixth, k=6",
        "query_features": torch.tensor([[[1.0, 0.0]]]),
        "query_labels": torch.tensor([[0]]),
        "candidate_features": top5_fail_candidates,
        "k": 6,
        "expected": {"top1_acc": 0.0, "top5_acc": 0.0},
    },
    {
        "name": "mixed queries average, k=6",
        "query_features": torch.stack(
            [
                torch.tensor([1.0, 0.0]),
                torch.tensor([0.9, 0.1]),
                torch.tensor([-1.0, 0.0]),
            ]
        ).unsqueeze(0),
        "query_labels": torch.tensor([[0, 0, 0]]),
        "candidate_features": mixed_candidates,
        "k": 6,
        "expected": {"top1_acc": 1 / 3, "top5_acc": 2 / 3},
    },
    {
        "name": "restricted k hides stronger later candidate, k=2",
        "query_features": torch.tensor([[[1.0, 0.0]]]),
        "query_labels": torch.tensor([[0]]),
        "candidate_features": torch.tensor(
            [
                [0.8, 0.6],
                [0.0, 1.0],
                [-1.0, 0.0],
                [1.0, 0.0],
            ]
        ),
        "k": 2,
        "expected": {"top1_acc": 1.0},
    },
    {
        "name": "full k exposes stronger later candidate, k=4",
        "query_features": torch.tensor([[[1.0, 0.0]]]),
        "query_labels": torch.tensor([[0]]),
        "candidate_features": torch.tensor(
            [
                [0.8, 0.6],
                [0.0, 1.0],
                [-1.0, 0.0],
                [1.0, 0.0],
            ]
        ),
        "k": 4,
        "expected": {"top1_acc": 0.0},
    },
]

for case in controlled_cases:
    run_controlled_kway_case(**case)